In [ ]:
# 1.1 Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from wordcloud import WordCloud

# For reproducibility
np.random.seed(42)



In [ ]:
# 1.2 Load the Dataset

# Load the datasets
fake_df = pd.read_csv('archive (2)/Fake.csv')
true_df = pd.read_csv('archive (2)/True.csv')

# Add a label column: 0 for Fake, 1 for True
fake_df['label'] = 0
true_df['label'] = 1

# Combine into a single DataFrame
df = pd.concat([fake_df, true_df], ignore_index=True)

# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)



In [ ]:
# 1.3 Show Dataset Structure

# Show columns
print("Columns:", df.columns.tolist())

# Show first 5 rows
display(df.head())

# Show class distribution
print("Class distribution:\n", df['label'].value_counts())


In [ ]:
# 2.1 Handle Missing Values

# Check missing values
missing = df.isnull().sum()
print("Missing values per column:\n", missing)

# Fill missing values in 'title' and 'text' with empty string (no 'author' column)
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')



In [ ]:
# 2.2 Combine Title and Text into a Single Feature (no 'author' column)
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

# Drop the original columns if you want to keep only 'content' and 'label'
# df = df[['content', 'label']]

df[['content', 'label']].head()


In [ ]:
# 2.3 Text Cleaning
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenize and remove stopwords
    words = [word for word in text.split() if word not in stop_words]
    # Lemmatize
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

# Apply cleaning (this may take a few minutes)
df['clean_content'] = df['content'].apply(clean_text)
df[['clean_content', 'label']].head()


In [ ]:
# 3.1 Class Balance Check
sns.countplot(x='label', data=df)
plt.title('Class Distribution (0=Fake, 1=True)')
plt.show()

print(df['label'].value_counts())


In [ ]:
# 3.2 Word Frequency Analysis
from collections import Counter

def get_top_n_words(corpus, n=None):
    words = ' '.join(corpus).split()
    counter = Counter(words)
    return counter.most_common(n)

# Top 20 words in fake news
fake_words = get_top_n_words(df[df['label']==0]['clean_content'], 20)
print('Top 20 words in Fake News:', fake_words)

# Top 20 words in real news
real_words = get_top_n_words(df[df['label']==1]['clean_content'], 20)
print('Top 20 words in Real News:', real_words)


In [ ]:
# 3.3 Word Clouds for Visualization
plt.figure(figsize=(12,6))
wordcloud_fake = WordCloud(width=600, height=400, background_color='white').generate(' '.join(df[df['label']==0]['clean_content']))
plt.subplot(1,2,1)
plt.imshow(wordcloud_fake, interpolation='bilinear')
plt.axis('off')
plt.title('Fake News Word Cloud')

wordcloud_real = WordCloud(width=600, height=400, background_color='white').generate(' '.join(df[df['label']==1]['clean_content']))
plt.subplot(1,2,2)
plt.imshow(wordcloud_real, interpolation='bilinear')
plt.axis('off')
plt.title('Real News Word Cloud')
plt.show()


In [ ]:
# 4.1 TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Use only the cleaned content
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['clean_content']).toarray()
y = df['label'].values

print('TF-IDF matrix shape:', X.shape)


In [ ]:
# 5.1 Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)


In [ ]:
# 5.2 Baseline Model: Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


In [ ]:
# 5.3 Baseline Model: Naive Bayes
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print('Naive Bayes Accuracy:', accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


In [ ]:
# 5.4 Advanced Model: Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


In [ ]:
# 5.5 Advanced Model: XGBoost
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)

print('XGBoost Accuracy:', accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


In [ ]:
# 6.1 Confusion Matrix Visualization
def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.show()

plot_confusion(y_test, y_pred_lr, 'Logistic Regression Confusion Matrix')
plot_confusion(y_test, y_pred_nb, 'Naive Bayes Confusion Matrix')
plot_confusion(y_test, y_pred_rf, 'Random Forest Confusion Matrix')
plot_confusion(y_test, y_pred_xgb, 'XGBoost Confusion Matrix')


In [ ]:
# 6.2 ROC-AUC Scores
from sklearn.metrics import roc_auc_score

print('Logistic Regression ROC-AUC:', roc_auc_score(y_test, y_pred_lr))
print('Naive Bayes ROC-AUC:', roc_auc_score(y_test, y_pred_nb))
print('Random Forest ROC-AUC:', roc_auc_score(y_test, y_pred_rf))
print('XGBoost ROC-AUC:', roc_auc_score(y_test, y_pred_xgb))


In [ ]:
# 7.1 Hyperparameter Tuning Example: Logistic Regression
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs']
}
gs_lr = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=3, scoring='f1', n_jobs=-1)
gs_lr.fit(X_train, y_train)
print('Best Params:', gs_lr.best_params_)
print('Best F1 Score:', gs_lr.best_score_)

y_pred_gs = gs_lr.predict(X_test)
print('Tuned Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_gs))
print(classification_report(y_test, y_pred_gs))


In [ ]:
# 8.1 Save the Best Model
import joblib

# Save the tuned logistic regression model
joblib.dump(gs_lr.best_estimator_, 'best_fake_news_model.pkl')

# Save the TF-IDF vectorizer
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')


In [ ]:
# 9.1 Feature Importance (Logistic Regression)
feature_names = tfidf.get_feature_names_out()
coefs = gs_lr.best_estimator_.coef_[0]

# Top 20 positive and negative features
sorted_idx = np.argsort(coefs)
print('Top 20 words for REAL news:')
for idx in sorted_idx[-20:][::-1]:
    print(f'{feature_names[idx]}: {coefs[idx]:.4f}')

print('\nTop 20 words for FAKE news:')
for idx in sorted_idx[:20]:
    print(f'{feature_names[idx]}: {coefs[idx]:.4f}')
